# Analiza wyników Colab i decyzja o następnym eksperymencie

Ten notebook powstał po pierwszym wyniku z Colaba (`unclip_mole_generation_smoke`). Jego zadanie jest praktyczne: wczytać wyniki z Colaba, rozpoznać czy mamy tylko smoke test czy pełną generację, porównać UnCLIP z lokalnym VAE baseline i jasno powiedzieć, co odpalać dalej.

Możesz go uruchamiać po każdym kolejnym ZIP-ie z Drive. Jeśli pojawi się `unclip_mole_generation_full`, notebook automatycznie uwzględni pełny wynik.

## Wniosek po aktualnie otrzymanym ZIP-ie

Otrzymany folder zawierał `unclip_mole_generation_smoke`, czyli tylko 2 obrazy testowe. Pipeline działa, ale smoke test nie pobił VAE: UnCLIP EEG miał około `SSIM = 0.069`, oracle około `SSIM = 0.096`, a wcześniejszy VAE ensemble dla `mole` miał około `SSIM = 0.286` na 44 obrazach.

Najbliższy krok: doprowadzić do pełnego `unclip_mole_generation_full`, a potem dopiero traktować porównanie jako rozstrzygające.

In [ ]:
from pathlib import Path
import csv
import json
import math
import sys
import zipfile

import pandas as pd

try:
    from IPython.display import Image, Markdown, display
except ModuleNotFoundError:
    class Markdown(str):
        pass

    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename

        def __repr__(self):
            return f'<Image {self.filename}>'

    def display(value):
        text = str(value)
        encoding = sys.stdout.encoding or 'utf-8'
        print(text.encode(encoding, errors='replace').decode(encoding, errors='replace'))

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

PROJECT_ROOT = Path.cwd()
PARTICIPANT = 'mole'
DRIVE_ROOT = Path('/content/drive/MyDrive')

if Path('/content').exists() and not DRIVE_ROOT.exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Nie udało się automatycznie zamontować Google Drive:', exc)

VAE_BASELINE_FALLBACK = [
    {
        'participant': 'mole', 'images': 44.0,
        'ensemble_l1': 0.2355921593579379,
        'ensemble_psnr': 11.59615940397436,
        'ensemble_ssim': 0.2864757523956624,
    },
    {
        'participant': 'MEAN', 'images': 44.0,
        'ensemble_l1': 0.23837537166069853,
        'ensemble_psnr': 11.584600968794389,
        'ensemble_ssim': 0.2832920191047544,
    },
]

# Lokalny folder po rozpakowaniu ZIP-a z Colaba, folder pobrany z Drive albo wynikowy folder Drive w Colabie.
RESULT_ROOT_CANDIDATES = [
    PROJECT_ROOT / 'colab_results_20260626T204736Z_3_001',
    PROJECT_ROOT / 'wyniki colab',
    PROJECT_ROOT / 'colab_results',
    DRIVE_ROOT / 'biai' / 'results',
    DRIVE_ROOT / 'biai' / 'wyniki colab',
    DRIVE_ROOT / 'wyniki colab',
    DRIVE_ROOT / 'Colab Notebooks',
    DRIVE_ROOT / 'biai',
]

VAE_SWEEP_CSV = PROJECT_ROOT / 'vae_participant_sweep_no_abc_20260626' / 'participant_vae_generation_all_summary.csv'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('Szukam wyników w:')
for candidate in RESULT_ROOT_CANDIDATES:
    print(' -', candidate, 'OK' if candidate.exists() else 'brak')
print('VAE baseline CSV =', VAE_SWEEP_CSV, 'OK' if VAE_SWEEP_CSV.exists() else 'brak')

In [ ]:
def read_json(path):
    path = Path(path)
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding='utf-8'))


def unique_paths(paths, key_func=None):
    seen = set()
    result = []
    for path in paths:
        key = key_func(path) if key_func else str(path.resolve()) if path.exists() else str(path)
        if key not in seen:
            seen.add(key)
            result.append(path)
    return result


def collect_result_dirs(roots):
    generation = []
    retrieval = []
    eegnet = []
    for root in unique_paths([Path(path) for path in roots if Path(path).exists()]):
        generation.extend(path for path in root.rglob(f'unclip_{PARTICIPANT}_generation_*') if path.is_dir())
        retrieval.extend(path for path in root.rglob(f'unclip_{PARTICIPANT}_retrieval') if path.is_dir())
        eegnet.extend(path for path in root.rglob(f'eegnet_{PARTICIPANT}_colab') if path.is_dir())
    return (
        unique_paths(sorted(generation), key_func=lambda path: path.name),
        unique_paths(sorted(retrieval), key_func=lambda path: path.name),
        unique_paths(sorted(eegnet), key_func=lambda path: path.name),
    )


def zip_contains_results(zip_path):
    try:
        with zipfile.ZipFile(zip_path) as archive:
            names = archive.namelist()
    except zipfile.BadZipFile:
        return False
    return any(
        f'unclip_{PARTICIPANT}_generation_' in name or f'unclip_{PARTICIPANT}_retrieval' in name
        for name in names
    )


def extract_result_zips(search_roots):
    zip_paths = []
    for root in unique_paths([Path(path) for path in search_roots if Path(path).exists()]):
        zip_paths.extend(root.rglob('*.zip'))
    zip_paths = unique_paths(sorted(zip_paths), key_func=lambda path: f'{path.name}:{path.stat().st_size}')
    extracted = []
    extract_root = Path('/content/colab_results_analysis') if Path('/content').exists() else PROJECT_ROOT / 'colab_results_analysis'
    for zip_path in zip_paths:
        if zip_path.name in {'biai_unclip_assets.zip', 'biai_eeg_qc_0_0p8.zip'}:
            continue
        if not zip_contains_results(zip_path):
            continue
        target = extract_root / zip_path.stem
        marker = target / '.extracted_by_analysis_notebook'
        if not marker.exists():
            target.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path) as archive:
                archive.extractall(target)
            marker.write_text(str(zip_path), encoding='utf-8')
        print('Rozpakowany ZIP wyników:', zip_path, '->', target)
        extracted.append(target)
    return extracted


result_roots = [path for path in RESULT_ROOT_CANDIDATES if path.exists()]
generation_dirs, retrieval_dirs, eegnet_dirs = collect_result_dirs(result_roots)

if not generation_dirs and DRIVE_ROOT.exists():
    print('Nie znalazłem wyników w typowych lokalizacjach; szukam po całym MyDrive...')
    generation_dirs, retrieval_dirs, eegnet_dirs = collect_result_dirs(result_roots + [DRIVE_ROOT])

if not generation_dirs:
    extracted_roots = extract_result_zips(result_roots + ([DRIVE_ROOT] if DRIVE_ROOT.exists() else []))
    if extracted_roots:
        generation_dirs, retrieval_dirs, eegnet_dirs = collect_result_dirs(result_roots + extracted_roots)

print('Znalezione katalogi generacji:')
for path in generation_dirs:
    print(' -', path)
print('\nZnalezione katalogi EEG->CLIP retrieval:')
for path in retrieval_dirs:
    print(' -', path)
print('\nZnalezione katalogi EEGNet klasyfikacji:')
for path in eegnet_dirs:
    print(' -', path)

if not generation_dirs:
    visible_roots = [str(path) for path in result_roots]
    raise FileNotFoundError(
        'Nie znalazłem katalogu unclip_*_generation_*.\n'
        f'Sprawdzone katalogi: {visible_roots}\n'
        'Jeśli wynik jest na Drive, upewnij się, że Drive jest zamontowany i folder zawiera np. '
        'unclip_mole_generation_smoke albo unclip_mole_generation_full.'
    )

## Metryki generacji UnCLIP

`eeg` oznacza obraz generowany z embeddingu przewidzianego z EEG. `oracle` oznacza obraz generowany z prawdziwego CLIP embeddingu obrazu; to pokazuje sufit samego generatora.

In [ ]:
generation_records = []

for gen_dir in generation_dirs:
    summary = read_json(gen_dir / 'unclip_generation_summary.json')
    if summary is None:
        continue
    run_type = 'full' if gen_dir.name.endswith('_full') else 'smoke' if gen_dir.name.endswith('_smoke') else gen_dir.name
    for variant in ['eeg', 'oracle']:
        metrics = summary.get(variant) or {}
        if not metrics:
            continue
        generation_records.append({
            'folder': gen_dir.name,
            'run_type': run_type,
            'variant': variant,
            'images': summary.get('images'),
            'steps': summary.get('num_inference_steps'),
            'guidance_scale': summary.get('guidance_scale'),
            'l1': metrics.get('l1'),
            'mse': metrics.get('mse'),
            'psnr': metrics.get('psnr'),
            'ssim': metrics.get('ssim'),
            'path': str(gen_dir),
        })

generation_df = pd.DataFrame(generation_records)
if generation_df.empty:
    raise FileNotFoundError('Znalazłem katalogi generacji, ale bez unclip_generation_summary.json.')

display(generation_df.sort_values(['run_type', 'variant']).reset_index(drop=True))

## Jakość dekodera EEG → CLIP embedding

To jest osobna część problemu. Jeśli retrieval jest tylko lekko ponad losowy, generator nie dostaje stabilnej informacji o obrazie.

In [ ]:
retrieval_records = []

for retrieval_dir in retrieval_dirs:
    summary = read_json(retrieval_dir / 'retrieval_summary.json')
    if summary is None:
        continue
    test = summary.get('test') or {}
    retrieval_records.append({
        'folder': retrieval_dir.name,
        'best_epoch': summary.get('best_epoch'),
        'best_validation_top5': summary.get('best_validation_top5'),
        'test_top1': test.get('top1'),
        'test_top5': test.get('top5'),
        'test_top10': test.get('top10'),
        'chance_top1': test.get('chance_top1'),
        'chance_top5': test.get('chance_top5'),
        'chance_top10': test.get('chance_top10'),
        'median_rank': test.get('median_rank'),
        'mean_rank': test.get('mean_rank'),
        'category_top1': test.get('category_top1'),
        'samples': test.get('samples'),
        'candidates': test.get('candidates'),
        'path': str(retrieval_dir),
    })

retrieval_df = pd.DataFrame(retrieval_records)
if retrieval_df.empty:
    print('Nie znalazłem retrieval_summary.json. Pomijam tę część.')
else:
    display(retrieval_df.reset_index(drop=True))
    row = retrieval_df.iloc[0]
    print(f"Top-5 retrieval: {row['test_top5']:.2%} vs chance {row['chance_top5']:.2%}")

## Porównanie z lokalnym VAE baseline

Dla naszych danych VAE nie jest piękne wizualnie, ale daje stabilny punkt odniesienia liczbowego podobieństwa. Jeśli UnCLIP ma niższy SSIM także na pełnym wyniku, to sam generator semantyczny nie jest jeszcze dobrym kierunkiem dla tych prostych bodźców.

In [ ]:
comparison_records = []

if VAE_SWEEP_CSV.is_file():
    vae_df = pd.read_csv(VAE_SWEEP_CSV)
    vae_source = str(VAE_SWEEP_CSV)
else:
    print('Brak lokalnego CSV z VAE baseline:', VAE_SWEEP_CSV)
    print('Używam wpisanego fallbacku z wcześniejszego sweepu VAE.')
    vae_df = pd.DataFrame(VAE_BASELINE_FALLBACK)
    vae_source = 'embedded VAE baseline fallback from 2026-06-26 sweep'

vae_participant = vae_df[vae_df['participant'] == PARTICIPANT]
vae_mean = vae_df[vae_df['participant'] == 'MEAN']
if not vae_participant.empty:
    row = vae_participant.iloc[0]
    comparison_records.append({
        'model': f'VAE ensemble ({PARTICIPANT})',
        'images': row.get('images'),
        'l1': row.get('ensemble_l1'),
        'psnr': row.get('ensemble_psnr'),
        'ssim': row.get('ensemble_ssim'),
        'source': vae_source,
    })
if not vae_mean.empty:
    row = vae_mean.iloc[0]
    comparison_records.append({
        'model': 'VAE ensemble (mean participants)',
        'images': row.get('images'),
        'l1': row.get('ensemble_l1'),
        'psnr': row.get('ensemble_psnr'),
        'ssim': row.get('ensemble_ssim'),
        'source': vae_source,
    })

for _, row in generation_df.iterrows():
    comparison_records.append({
        'model': f"Stable UnCLIP {row['variant']} ({row['run_type']})",
        'images': row['images'],
        'l1': row['l1'],
        'psnr': row['psnr'],
        'ssim': row['ssim'],
        'source': row['path'],
    })

comparison_df = pd.DataFrame(comparison_records)
if not comparison_df.empty:
    for column in ['images', 'l1', 'psnr', 'ssim']:
        comparison_df[column] = pd.to_numeric(comparison_df[column], errors='coerce')
    display(comparison_df.sort_values('ssim', ascending=False).reset_index(drop=True))

## Podgląd gridów

Najbardziej ufamy metrykom dopiero na pełnym `full`, ale grid szybko pokazuje, czy generator łapie strukturę bodźca czy tylko styl kategorii.

In [ ]:
for gen_dir in generation_dirs:
    grid_dir = gen_dir / 'grids'
    grids = sorted(grid_dir.glob('*.jpg')) + sorted(grid_dir.glob('*.png'))
    if not grids:
        print('Brak gridów w', gen_dir)
        continue
    display(Markdown(f'### {gen_dir.name}'))
    for grid in grids:
        display(Markdown(f'`{grid.name}`'))
        display(Image(filename=str(grid)))

## Automatyczny werdykt

Ta komórka nie zastępuje oceny wzrokowej, ale pilnuje, żeby nie pomylić smoke testu z pełnym wynikiem.

In [ ]:
has_full = bool((generation_df['run_type'] == 'full').any())
has_smoke = bool((generation_df['run_type'] == 'smoke').any())
messages = []

if has_smoke and not has_full:
    messages.append('⚠️ Mamy tylko smoke test. To potwierdza, że pipeline działa, ale nie jest pełnym wynikiem eksperymentu.')
    messages.append('Następny ruch: uruchom ponownie `REKONSTRUKCJA_UNCLIP_COLAB.ipynb` z najnowszego brancha. Notebook powinien pominąć smoke i zrobić `unclip_mole_generation_full`.')
elif has_full:
    messages.append('✅ Jest pełny wynik `unclip_mole_generation_full`; można traktować porównanie jako właściwą ocenę eksperymentu.')

if not retrieval_df.empty:
    row = retrieval_df.sort_values('test_top5', ascending=False).iloc[0]
    top5_gain = row['test_top5'] - row['chance_top5']
    messages.append(f"Retrieval EEG→CLIP top-5: {row['test_top5']:.2%} vs losowo {row['chance_top5']:.2%}; przewaga = {top5_gain:.2%}.")
    if top5_gain < 0.10:
        messages.append('Sygnał EEG→CLIP jest realny, ale jeszcze słaby; sam generator dostaje dość rozmytą informację.')

if 'comparison_df' in globals() and not comparison_df.empty:
    vae_rows = comparison_df[comparison_df['model'].str.contains('VAE ensemble', na=False)]
    unclip_eeg_rows = comparison_df[comparison_df['model'].str.contains('Stable UnCLIP eeg', na=False)]
    if not vae_rows.empty and not unclip_eeg_rows.empty:
        best_vae_ssim = float(vae_rows['ssim'].max())
        best_unclip_ssim = float(unclip_eeg_rows['ssim'].max())
        messages.append(f'Najlepszy VAE SSIM: {best_vae_ssim:.3f}; najlepszy UnCLIP EEG SSIM: {best_unclip_ssim:.3f}.')
        if best_unclip_ssim < best_vae_ssim:
            messages.append('Na obecnych wynikach UnCLIP nie bije VAE. Następny sensowny eksperyment to poprawa dekodera EEG→embedding albo generator bardziej wierny prostym wzorcom.')
        else:
            messages.append('UnCLIP zaczyna bić VAE liczbowo; następny krok to analiza per-kategoria i stabilność między uczestnikami.')

display(Markdown('\n\n'.join(messages)))

## Decyzja o następnym eksperymencie

Na podstawie aktualnego smoke testu nie warto jeszcze eskalować w większy model generatywny. Najpierw potrzebujemy pełnego `unclip_mole_generation_full`. Jeśli pełny wynik nadal będzie wyraźnie pod VAE, następny notebook powinien iść w jedną z dwóch stron:

1. **Poprawa EEG → embedding**: dłuższy trening, inny loss, augmentacje, ensemble ResNet/DINO/CLIP i kalibracja po kategoriach.
2. **Generator wierniejszy bodźcom**: mniej semantyczny niż Stable UnCLIP, bardziej warunkowany strukturą prostych czarno-białych wzorców.

Mój obecny wybór: najpierw pełne UnCLIP `full`, potem jeśli wynik zostanie słaby — notebook z diagnostyką EEG→embedding i rerankingiem/nearest-neighbor jako mocnym, uczciwym baseline dla rekonstrukcji obrazu.